# Langfuse callback handler demo

Shows the Langfuse-callback-handler path: pass `LangfuseHook(client)` through
`config={"callbacks": [...]}` and every successful ingest emits a Langfuse
span with the per-ingest `IngestResult` plus a Mermaid snapshot of the graph.

**Requires** `agent_pinboard[langfuse]` and Langfuse credentials in env
(`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`).


## 0. Imports + Langfuse client

In [ ]:
from __future__ import annotations

import os
from datetime import UTC, datetime

from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel

from agent_pinboard import Entity, make_graph_tools, node, pin
from agent_pinboard.integrations.langfuse_hook import LangfuseHook

from langfuse import Langfuse
client = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)
handler = LangfuseHook(client)


## 1. Tiny domain — one tool that produces one fact per call

In [ ]:
IP = Entity(name="IP", description="ipv4/ipv6", normalizer=lambda v: str(v).lower())

class Lookup(BaseModel):
    queried: str = node(type=IP, description="queried IP")

@pin(model=Lookup)
@tool
def lookup(value: str, runtime: ToolRuntime) -> dict:
    """Mock IP lookup."""
    return {"queried": value}


## 2. Drive a single tool call with the Langfuse handler attached

(In a real agent setup `handler` lives next to your other LangChain callbacks
in `config["callbacks"]`; here we go through `BaseTool.invoke` directly to keep
the example minimal.)

In [ ]:
store = InMemoryStore()
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list, add_messages]

g = StateGraph(State)
g.add_node("tools", ToolNode([lookup]))
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile(store=store)

graph.invoke(
    {"messages": [AIMessage(
        content="",
        tool_calls=[{"name": "lookup", "args": {"value": "8.8.8.8"},
                     "id": "c-1", "type": "tool_call"}],
    )]},
    config={
        "configurable": {"thread_id": "langfuse-demo"},
        "callbacks": [handler],
    },
)
client.flush()
print("Check your Langfuse dashboard — there should be one\n"
      "`agent_pinboard.ingest` span and one `agent_pinboard.graph_snapshot` span.")
